# 16. Support Vector Machine (SVM)
**Use case:** Oil & Gas — Lithology classification from well logs

Every new well produces logs of gamma ray, bulk density, neutron porosity and resistivity every half-foot, but rock type is only confirmed where expensive core is taken. Geologists spend days hand-interpreting lithology, and inconsistent picks between interpreters slow reservoir models and completion decisions. Using cored wells as labelled data, an SVM can produce a consistent Sandstone / Shale / Limestone track for every new well within minutes. Success means high accuracy on a blind well the model never saw, calibrated confidence, and only the uncertain intervals sent to a petrophysicist.

> 💡 Imagine painting the widest possible road between two groups of houses so neither side's houses touch the road. Only the houses right at the road edge matter for where you paint it - those are the support vectors. A kernel is like letting the road bend around neighbourhoods instead of forcing it to be straight.

Legend in code: `# 🐍 PY:` Python lesson · `# 🧠 ML:` ML lesson

In [ ]:
# Run once (Colab already has most of these)
!pip -q install numpy pandas scikit-learn scipy xgboost torch

## EASY: Draw the line between Sandstone and Shale
Fit a linear SVM on 12 hand-typed depth samples and see which samples become support vectors and how wide the margin is.

| 5W + H | |
|---|---|
| **Who** | A junior petrophysicist learning how a machine can read two well logs the way a geologist eyeballs a crossplot. |
| **What** | Classify depth samples as Sandstone or Shale using gamma ray (API) and bulk density (g/cc). |
| **When** | Right after logging a new well, before the geologist does a detailed interpretation. |
| **Where** | A notebook on the petrophysicist's laptop with a handful of samples typed in. |
| **Why** | Shale is radioactive (high gamma ray) and denser; a straight line on the GR-density crossplot often separates the two. |
| **How** | Scale the two logs, fit SVC(kernel='linear'), print support vectors, weights, margin width and the decision score for three new depths. |

**🧠 ML concepts:** Maximum-margin hyperplane, Support vectors, Margin width = 2/||w||, Decision function sign and magnitude, Why feature scaling matters for SVM  
**🐍 Python concepts:** List of tuples, List comprehension with tuple unpacking, Conditional expression, numpy mean/std along an axis, f-string format specs

In [ ]:
# Easy: separate Sandstone from Shale with a straight line (linear SVM)
import numpy as np
from sklearn.svm import SVC

# 🐍 PY: a list of tuples - each tuple is one depth sample: (gamma_ray_API, density_g_cc, label)
samples = [
    (35, 2.30, "Sandstone"), (40, 2.32, "Sandstone"), (45, 2.28, "Sandstone"),
    (50, 2.35, "Sandstone"), (38, 2.25, "Sandstone"), (55, 2.33, "Sandstone"),
    (95, 2.52, "Shale"), (110, 2.55, "Shale"), (120, 2.50, "Shale"),
    (88, 2.48, "Shale"), (130, 2.58, "Shale"), (100, 2.45, "Shale"),
]
# 🐍 PY: list comprehension pulls the first two numbers of each tuple into a 2-D numpy array
X = np.array([[gr, rho] for gr, rho, _ in samples])
# 🐍 PY: conditional expression inside a comprehension -> 1 for Shale, 0 for Sandstone
y = np.array([1 if lab == "Shale" else 0 for _, _, lab in samples])

# Put both features on a similar scale: GR is ~100s, density is ~2.x
# 🧠 ML: SVM measures distances, so a feature with big numbers would dominate; we rescale by hand
X_scaled = (X - X.mean(axis=0)) / X.std(axis=0)

# 🧠 ML: kernel="linear" means the boundary is a straight line; C controls how hard we punish mistakes
model = SVC(kernel="linear", C=1.0)
model.fit(X_scaled, y)

# 🧠 ML: support vectors are the few samples sitting closest to the line - they alone define it
print("Support vector indices:", model.support_)
for i in model.support_:
    gr, rho, lab = samples[i]
    # 🐍 PY: f-string with format specs (:>4 right-align, :.2f two decimals)
    print(f"  sample {i:>2}: GR={gr:>4} API  RHOB={rho:.2f}  -> {lab}")

# 🧠 ML: the line is w . x + b = 0 ; the margin (street width) is 2 / ||w||
w, b = model.coef_[0], model.intercept_[0]
print(f"\nWeights w = {np.round(w, 3)}, bias b = {b:.3f}")
print(f"Margin width (scaled units) = {2 / np.linalg.norm(w):.3f}")

# Score a new depth: decision value sign tells the side of the line
new_depths = [(60, 2.36), (85, 2.44), (115, 2.53)]
for gr, rho in new_depths:
    # 🐍 PY: build a 1-row array and scale it with the SAME mean/std as training
    x = (np.array([[gr, rho]]) - X.mean(axis=0)) / X.std(axis=0)
    score = model.decision_function(x)[0]
    # 🧠 ML: |score| < 1 means the point falls inside the margin - the model is less sure
    label = "Shale" if score > 0 else "Sandstone"
    zone = "inside margin" if abs(score) < 1 else "clear side"
    print(f"GR={gr:>3} RHOB={rho:.2f} -> score {score:+.2f} -> {label:<9} ({zone})")

**Try this next:**
- Change C to 0.01 and then 100 and see how many support vectors appear and how the margin width changes.
- Add a sandstone sample with GR=90 (a 'hot' radioactive sand) and refit - which samples become support vectors now?
- Remove the scaling line and fit on raw X - compare the weights and the predictions.

## MEDIUM: RBF SVM on 1,800 log samples, three rock types
Train an RBF-kernel SVC on four standardised well logs to classify Sandstone, Shale and Limestone and read its confusion matrix.

| 5W + H | |
|---|---|
| **Who** | A petrophysics team that wants a first-pass lithology log for every new well in a field. |
| **What** | A three-class classifier using gamma ray, density, neutron porosity and log10 resistivity. |
| **When** | After each well is logged, to produce a quick lithology track for the geologist to review. |
| **Where** | A Python script run by the subsurface data team on exported log data. |
| **Why** | Rock types overlap on any single log; a curved (RBF) boundary in four dimensions separates them far better than a cutoff on one curve. |
| **How** | Generate synthetic logs per rock type, split with stratify, fit StandardScaler on train only, compare scaled vs unscaled RBF SVC, then print a classification report and confusion matrix. |

**🧠 ML concepts:** RBF kernel and the gamma parameter, Multi-class SVM (one-vs-one inside SVC), Fit scaler on training data only (no leakage), Stratified split, Precision, recall, F1 and confusion matrix  
**🐍 Python concepts:** Nested dict of tuples, Dict comprehension, dict.items() looping, pd.concat with ignore_index, Tuple of booleans to drive a loop, DataFrame for labelled confusion matrix

In [ ]:
# Medium: RBF SVM classifying Sandstone / Shale / Limestone from four well logs
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

np.random.seed(42)
N_PER_CLASS = 600

# 🐍 PY: a dict of dicts holds typical (mean, std) log responses per rock type
profiles = {
    "Sandstone": {"gamma_ray_api": (55, 20), "density_g_cc": (2.38, 0.09),
                  "neutron_porosity": (0.20, 0.06), "resistivity_ohm_m": (20, 12)},
    "Shale":     {"gamma_ray_api": (95, 25), "density_g_cc": (2.48, 0.08),
                  "neutron_porosity": (0.30, 0.07), "resistivity_ohm_m": (5, 4)},
    "Limestone": {"gamma_ray_api": (40, 18), "density_g_cc": (2.58, 0.08),
                  "neutron_porosity": (0.10, 0.06), "resistivity_ohm_m": (40, 30)},
}

frames = []
# 🐍 PY: .items() yields (key, value) pairs so we loop over rock name and its profile together
for rock, cols in profiles.items():
    # 🐍 PY: dict comprehension draws N random values per column from a normal distribution
    data = {c: np.random.normal(m, s, N_PER_CLASS) for c, (m, s) in cols.items()}
    df_rock = pd.DataFrame(data)
    df_rock["lithology"] = rock
    frames.append(df_rock)
# 🐍 PY: pd.concat stacks the three tables; ignore_index renumbers rows 0..N-1
df = pd.concat(frames, ignore_index=True)
# 🧠 ML: resistivity is skewed (log-normal in reality), so model log10 of it
df["resistivity_ohm_m"] = df["resistivity_ohm_m"].clip(lower=0.5)
df["log_resistivity"] = np.log10(df["resistivity_ohm_m"])

features = ["gamma_ray_api", "density_g_cc", "neutron_porosity", "log_resistivity"]
# 🐍 PY: .values turns DataFrame columns into a plain numpy array
X, y = df[features].values, df["lithology"].values
print("Rows per class:\n", df["lithology"].value_counts().to_string(), "\n")

# 🧠 ML: stratify=y keeps the same rock mix in train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

# 🧠 ML: fit the scaler on TRAIN only, then apply to test - avoids leaking test info
scaler = StandardScaler().fit(X_train)
X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)

# 🧠 ML: RBF kernel bends the boundary; gamma = how local each point's influence is
for scaled in (False, True):
    Xtr, Xte = (X_train_s, X_test_s) if scaled else (X_train, X_test)
    clf = SVC(kernel="rbf", C=1.0, gamma="scale").fit(Xtr, y_train)
    acc = accuracy_score(y_test, clf.predict(Xte))
    print(f"RBF SVM  scaled={str(scaled):<5}  test accuracy = {acc:.3f}")

model = SVC(kernel="rbf", C=1.0, gamma="scale").fit(X_train_s, y_train)
# 🧠 ML: n_support_ shows how many training samples each class needs to define the boundary
y_pred = model.predict(X_test_s)
print(f"\nSupport vectors per class: {dict(zip(model.classes_, model.n_support_.tolist()))}")
print("\nClassification report (test):")
print(classification_report(y_test, y_pred, digits=3))

# 🐍 PY: wrap the numpy confusion matrix in a DataFrame for readable row/column labels
cm = pd.DataFrame(confusion_matrix(y_test, y_pred, labels=model.classes_),
                  index=[f"true_{c}" for c in model.classes_],
                  columns=[f"pred_{c}" for c in model.classes_])
print("Confusion matrix:\n", cm.to_string())

# Business interpretation
shale_as_sand = cm.loc["true_Shale", "pred_Sandstone"]
print(f"\nShale called Sandstone: {shale_as_sand} samples."
      " These are the costly errors - a petrophysicist might flag a non-reservoir zone as pay.")

**Try this next:**
- Try gamma=0.01 and gamma=10 in the final SVC and watch accuracy and support-vector counts change.
- Drop log_resistivity from the features list - which class gets worse?
- Make Shale and Sandstone GR means closer (e.g. 70 and 85) to simulate shaly sands and see the confusion grow.

## HARD: Tuned, calibrated SVM tested on a blind well
Build a production lithology pipeline with grid-searched C and gamma, calibrated probabilities, a blind-well test in depth order, permutation importance and a review threshold.

| 5W + H | |
|---|---|
| **Who** | The subsurface data science team supporting geologists across a multi-well field development. |
| **What** | A deployable pipeline (log transform, scaler, RBF SVM, Platt calibration) that labels every depth sample with a rock type and a confidence. |
| **When** | Run automatically when a new well's logs land in the data store; low-confidence intervals are routed to a petrophysicist. |
| **Where** | Batch scoring service fed by the well-log repository, with the model artifact versioned in the model registry. |
| **Why** | Random splits over-state accuracy because neighbouring depths are near-duplicates; only a blind well tells you how the model will behave in a new hole. |
| **How** | Simulate five wells of stacked beds, hold out well W5, GridSearchCV over C and gamma with stratified folds, calibrate with CalibratedClassifierCV, score W5, explain with permutation importance and choose a confidence threshold. |

**🧠 ML concepts:** Pipeline to prevent leakage inside CV, Grid search over C and gamma, Probability calibration (Platt scaling), Blind-well (group) validation vs random split, Permutation importance, Confidence threshold for human review  
**🐍 Python concepts:** Functions with docstrings, Star-unpacking into a list, FunctionTransformer wrapping a plain function, pivot_table for reshaping, sorted with key=lambda, os.path and tempfile for saving artifacts

In [ ]:
# Hard: production lithology classifier - tuned RBF SVM, calibrated, tested on a blind well
import os
import tempfile
import warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.calibration import CalibratedClassifierCV
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, f1_score, log_loss, confusion_matrix
from sklearn.dummy import DummyClassifier

warnings.filterwarnings("ignore")
RNG = np.random.default_rng(42)
FEATURES = ["gamma_ray_api", "density_g_cc", "neutron_porosity", "resistivity_ohm_m"]
PROFILE = {  # (mean, std) per log for each rock type
    "Sandstone": [(55, 20), (2.38, 0.09), (0.20, 0.06), (20, 12)],
    "Shale":     [(95, 25), (2.48, 0.08), (0.30, 0.07), (5, 4)],
    "Limestone": [(40, 18), (2.58, 0.08), (0.10, 0.06), (40, 30)],
}


def simulate_well(name, n_beds=25, top_ft=5000.0):
    """Return a depth-ordered log for one well made of stacked rock beds."""
    # 🐍 PY: multiple assignment sets two variables in one line
    rows, depth = [], top_ft
    for _ in range(n_beds):
        rock = RNG.choice(list(PROFILE), p=[0.35, 0.45, 0.20])
        # 🧠 ML: neighbouring samples in a bed are correlated - this is why random splits are too optimistic
        thickness = int(RNG.integers(5, 30))
        # 🧠 ML: a bed-level shift mimics real geology - the same rock reads slightly differently bed to bed
        bias = [RNG.normal(0, s * 0.3) for _, s in PROFILE[rock]]
        for _ in range(thickness):
            vals = [RNG.normal(m + b, s) for (m, s), b in zip(PROFILE[rock], bias)]
            rows.append([name, depth, *vals, rock])
            depth += 0.5
    # 🐍 PY: *vals unpacks the list into the row so all columns sit at one level
    return pd.DataFrame(rows, columns=["well", "depth_ft", *FEATURES, "lithology"])


def log_resistivity(X):
    """Resistivity spans decades, so take log10 of the last column."""
    X = np.array(X, dtype=float, copy=True)
    # 🐍 PY: X[:, 3] selects every row of column 3 (resistivity)
    X[:, 3] = np.log10(np.clip(X[:, 3], 0.5, None))
    return X


def build_pipeline():
    # 🧠 ML: putting transform + scaler + SVM in one Pipeline means CV refits scaling on each fold (no leakage)
    return Pipeline([
        ("logres", FunctionTransformer(log_resistivity)),
        ("scale", StandardScaler()),
        ("svm", SVC(kernel="rbf", class_weight="balanced", random_state=42)),
    ])


# 1. Data: four training wells, one blind well kept completely aside
wells = pd.concat([simulate_well(f"W{i}") for i in range(1, 6)], ignore_index=True)
# 🧠 ML: hold out a WHOLE well - the honest way to test a model meant for new wells
train = wells[wells.well != "W5"].sort_values(["well", "depth_ft"])
blind = wells[wells.well == "W5"].sort_values("depth_ft")
print(f"Train rows {len(train)} from wells {sorted(train.well.unique())}, blind well W5 rows {len(blind)}")
# 🐍 PY: dict comprehension converts pandas keys/values to plain str/float for clean printing
mix = train.lithology.value_counts(normalize=True).round(2)
print("Train mix:", {str(k): float(v) for k, v in mix.items()})

# 2. Grid search C and gamma with stratified 5-fold CV
# 🧠 ML: C = penalty for margin violations (high C = tighter fit); gamma = reach of each support vector
grid = {"svm__C": [0.3, 1, 10, 50], "svm__gamma": [0.03, 0.1, 0.3, 1.0]}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# 🧠 ML: macro-F1 averages F1 over rock types, so the rarer Limestone counts as much as Shale
search = GridSearchCV(build_pipeline(), grid, cv=cv, scoring="f1_macro", n_jobs=1)
search.fit(train[FEATURES], train.lithology)
print(f"\nBest params {search.best_params_}  CV macro-F1 {search.best_score_:.3f}")
res = pd.DataFrame(search.cv_results_)
# 🐍 PY: pivot_table reshapes the long results into a C x gamma grid for easy reading
print(res.pivot_table(index="param_svm__C", columns="param_svm__gamma",
                      values="mean_test_score").round(3).to_string())

# 3. Probability calibration
# 🧠 ML: SVM scores are not probabilities; sigmoid (Platt) calibration maps them to trustworthy 0-1 values
best_pipe = search.best_estimator_
calibrated = CalibratedClassifierCV(best_pipe, method="sigmoid", cv=5)
calibrated.fit(train[FEATURES], train.lithology)

# 4. Blind-well test - the honest estimate of performance in a new well
# 🧠 ML: a 'always predict the majority rock' baseline shows how much the SVM really adds
baseline = DummyClassifier(strategy="most_frequent").fit(train[FEATURES], train.lithology)
proba = calibrated.predict_proba(blind[FEATURES])
# 🐍 PY: argmax(axis=1) finds the most likely class column for every row
pred = calibrated.classes_[proba.argmax(axis=1)]
print("\nBlind well W5 results:")
for name, p in [("Baseline (always majority)", baseline.predict(blind[FEATURES])), ("Calibrated SVM", pred)]:
    print(f"  {name:<27} acc {accuracy_score(blind.lithology, p):.3f}  "
          f"macro-F1 {f1_score(blind.lithology, p, average='macro'):.3f}")
print(f"  Calibrated log-loss {log_loss(blind.lithology, proba, labels=calibrated.classes_):.3f}")
cm = confusion_matrix(blind.lithology, pred, labels=calibrated.classes_)
print(pd.DataFrame(cm, index=calibrated.classes_, columns=calibrated.classes_).to_string())

# 5. Explainability via permutation importance on the blind well
# 🧠 ML: shuffle one log at a time; the bigger the F1 drop, the more the model relies on that log
imp = permutation_importance(best_pipe, blind[FEATURES], blind.lithology, scoring="f1_macro",
                             n_repeats=5, random_state=42)
print("\nPermutation importance (drop in macro-F1):")
for f, m in sorted(zip(FEATURES, imp.importances_mean), key=lambda t: -t[1]):
    print(f"  {f:<18} {m:.3f} " + "#" * int(m * 100))

# 6. Business decision: send low-confidence depths to a petrophysicist
# 🧠 ML: calibrated max-probability is a confidence; a threshold trades review effort for error risk
# 🐍 PY: assign returns a copy of the DataFrame with new columns added
blind = blind.assign(pred=pred, conf=proba.max(axis=1))
print("\nThreshold | auto-labelled | errors in auto | depths to review")
for t in [0.5, 0.7, 0.8, 0.9]:
    auto = blind[blind.conf >= t]
    errs = int((auto.pred != auto.lithology).sum())
    print(f"   {t:.1f}    |   {len(auto):>4}        |   {errs:>3}          |   {len(blind) - len(auto):>4}")

# 7. Depth-ordered strip - 12 samples around the first misclassified depth
first_miss = int(np.argmax((blind.pred != blind.lithology).values))
print("\nDepth log (blind well, around first miss):")
for _, r in blind.iloc[max(first_miss - 2, 0): first_miss + 2].iterrows():
    flag = "" if r.pred == r.lithology else "  <-- miss"
    print(f"  {r.depth_ft:7.1f} ft  true {r.lithology:<9} pred {r.pred:<9} conf {r.conf:.2f}{flag}")

# 8. Save the model for deployment
out = os.path.join(tempfile.mkdtemp(), "lithology_svm.joblib")
# 🧠 ML: save the WHOLE calibrated pipeline so scoring repeats the exact same preprocessing
joblib.dump(calibrated, out)
print(f"\nSaved calibrated model -> {os.path.basename(out)}")
print("Model-risk note: validate on blind wells per basin; document logs, tool vendors and"
      " depth ranges used; re-calibrate when a new field or logging suite is added.")

**Try this next:**
- Replace the StratifiedKFold with GroupKFold over well names and compare the CV score with the blind-well score.
- Change calibration method to 'isotonic' and compare the blind-well log-loss.
- Add a fourth class 'Dolomite' to PROFILE (density about 2.75) and rerun the whole pipeline.